# V5W_03 — DHSLP Subject-Specific (5 parole)

Adattato da **EEG_13b** (versione snella: senza Optuna/t-SNE/confronti). LOSO per soggetto: test=ultima sessione, val=penultima, train=resto. **5 classi** (acqua/aiuto/mangiare/no/si), **chance 20%**. Grafi da `data/5words_subjects/graphs/`.

**Env: `daniele_311`**, GPU consigliata.

## §1 — Config

In [ ]:
import json, logging, re, traceback
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score, recall_score
from tqdm.auto import tqdm
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w03')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'v5w03'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ---- CONFIG (5 parole, chance 20%) ----
N_CHANNELS, N_SAMPLES = 61, 384
N_CLASSES   = 5
DATA_METRIC = 'abs_pcc'
# nomi classe = le 5 parole (ordine idx 0-4)
_i2l = json.loads((project_root/'configs'/'label_schemes'/'idx2label_5words.json').read_text())
CLASS_NAMES = [_i2l[str(i)] for i in range(N_CLASSES)]

# DHSLP iperparametri (identici a EEG_13b)
K_WINDOWS, N_EDGES, D_MODEL, HIDDEN, N_LAYERS, DROPOUT = 8, 16, 64, 128, 2, 0.5
LR, WEIGHT_DECAY, GRAD_CLIP, BATCH_SIZE = 1e-3, 1e-2, 1.0, 32
MAX_EPOCHS, PATIENCE = 200, 70
USE_INSTANCE_NORM, LABEL_SMOOTHING, MIXUP_ALPHA = True, 0.1, 0.4
GAP_THRESHOLD, GAP_PATIENCE = 0.25, 15
USE_AUGMENTATION = True
AUG_NOISE_STD, AUG_AMP_RANGE, AUG_SHIFT_MAX, AUG_MASK_LEN = 0.05, (0.85, 1.15), 15, 20
T_WIN = N_SAMPLES // K_WINDOWS

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

# ---- Grafi 5words ISOLATI (no collisione con la tesi) ----
HG_ROOT = project_root / 'data' / '5words_subjects' / 'graphs' / f'hypergraphs_pruned_{DATA_METRIC}'
assert HG_ROOT.exists(), f'Grafi non trovati: {HG_ROOT} — esegui prima V5W_02'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
subj_sess = defaultdict(lambda: defaultdict(list))
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m: subj_sess[int(m.group(1))][int(m.group(2))].append(p)
ALL_SUBJ = sorted(subj_sess.keys())
log.info(f'HG_ROOT: {HG_ROOT}')
log.info(f'Soggetti: {len(ALL_SUBJ)}  classi: {CLASS_NAMES}  chance={1/N_CLASSES:.0%}')


## §2 — Dataset LOSO (y = parola 0-4, nessun remap)

In [ ]:
def _augment_eeg(x):
    x = x.clone()
    if torch.rand(1) < 0.5: x = x + torch.randn_like(x) * AUG_NOISE_STD
    if torch.rand(1) < 0.5: x = x * torch.empty(1).uniform_(*AUG_AMP_RANGE)
    if torch.rand(1) < 0.5:
        shift = torch.randint(-AUG_SHIFT_MAX, AUG_SHIFT_MAX + 1, (1,)).item()
        x = torch.roll(x, shift, dims=1)
    if torch.rand(1) < 0.3:
        T = x.shape[1]; start = torch.randint(0, max(1, T - AUG_MASK_LEN), (1,)).item()
        x[:, start:start + AUG_MASK_LEN] = 0.0
    if torch.rand(1) < 0.2:
        ch = torch.randint(0, x.shape[0], (1,)).item(); x[ch] = 0.0
    return x

class EEGRawDatasetSS(Dataset):
    def __init__(self, items, use_instance_norm=True, augment=False):
        self.items, self.use_instance_norm, self.augment = items, use_instance_norm, augment
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        d = torch.load(p, weights_only=False)
        x = d['x'].float()
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        if self.augment: x = _augment_eeg(x)
        return x, torch.tensor(label, dtype=torch.long)

def _collect(subj_id, sess_list):
    """5words: y nel .pt e' GIA' la parola 0-4 (label2idx_5words). Nessun remap."""
    items = []
    for s in sess_list:
        for p in subj_sess[subj_id][s]:
            d = torch.load(p, weights_only=False)
            y = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            if 0 <= y < N_CLASSES: items.append((p, y))
    return items

def make_loso_loaders(subj_id):
    sids = sorted(subj_sess[subj_id].keys())
    if len(sids) < 2: return None
    test_sess, val_sess = sids[-1], sids[-2]
    train_sess = [s for s in sids if s not in (test_sess, val_sess)]
    tr_items = _collect(subj_id, train_sess)
    va_items = _collect(subj_id, [val_sess])
    te_items = _collect(subj_id, [test_sess])
    if not tr_items or not te_items: return None
    tr_labels = np.array([it[1] for it in tr_items])
    counts = np.bincount(tr_labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / np.clip(counts[tr_labels], 1, None), dtype=torch.float)
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    kw = dict(num_workers=0, pin_memory=False)
    return (DataLoader(EEGRawDatasetSS(tr_items, USE_INSTANCE_NORM, USE_AUGMENTATION), BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(EEGRawDatasetSS(va_items, USE_INSTANCE_NORM, False), BATCH_SIZE, shuffle=False, **kw),
            DataLoader(EEGRawDatasetSS(te_items, USE_INSTANCE_NORM, False), BATCH_SIZE, shuffle=False, **kw))


## §3 — Modello DHSLP

In [ ]:
class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)
    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6); d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv = (1.0 / d_v.sqrt()).unsqueeze(-1); De = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out
        out = torch.bmm(H, out); out = Dv * out
        if self.bias is not None: out = out + self.bias
        return out

class DHSLP(nn.Module):
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS, n_edges=N_EDGES,
                 d_model=D_MODEL, hidden=HIDDEN, n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(nn.Linear(T_win, d_model), nn.LayerNorm(d_model), nn.ELU())
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout); self.clf = nn.Linear(hidden, n_classes)
    def build_dynamic_H(self, feat):
        scores = torch.matmul(feat, self.E.T) / (self.d_model ** 0.5)
        return torch.softmax(scores, dim=2)
    def forward(self, x):
        B, N, T = x.shape; outs = []
        for k in range(self.K):
            x_k = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc
            H_k = self.build_dynamic_H(feat); out = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out); out = self.drop(out)
            outs.append(out.mean(dim=1))
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_m = DHSLP(); log.info(f'device={device}  DHSLP {sum(p.numel() for p in _m.parameters()):,} params'); del _m


## §4 — Train / Eval

In [ ]:
_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def run_epoch(model, loader, optimizer=None, mixup_alpha=0.0):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train and mixup_alpha > 0:
                lam = float(np.random.beta(mixup_alpha, mixup_alpha))
                idx = torch.randperm(len(x), device=x.device)
                x = lam * x + (1 - lam) * x[idx]
                logits = model(x)
                loss = lam * _criterion(logits, y) + (1 - lam) * _criterion(logits, y[idx])
            else:
                logits = model(x); loss = _criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); optimizer.step()
            total_loss += loss.item() * len(y)
            all_labels.extend(y.cpu().numpy()); all_preds.extend(logits.argmax(1).cpu().numpy())
    bacc = balanced_accuracy_score(all_labels, all_preds)
    return total_loss / len(loader.dataset), bacc, np.array(all_labels), np.array(all_preds)

def train_subject(subj_id, tr_l, va_l, te_l):
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=f'v5w03_P{subj_id:03d}_5words',
                     config=dict(notebook='V5W_03', model='DHSLP_SS_5words', subject=f'P{subj_id:03d}',
                                 n_classes=N_CLASSES, k_windows=K_WINDOWS, n_edges=N_EDGES, d_model=D_MODEL,
                                 hidden=HIDDEN, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
                                 max_epochs=MAX_EPOCHS, patience=PATIENCE, mixup_alpha=MIXUP_ALPHA,
                                 n_train=len(tr_l.dataset)),
                     reinit='finish_previous', settings=wandb.Settings(start_method='thread'))
    model = DHSLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, patience_cnt, gap_cnt = 0.0, None, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_b, _, _ = run_epoch(model, tr_l, opt, mixup_alpha=MIXUP_ALPHA)
        va_loss, va_b, _, _ = run_epoch(model, va_l)
        sched.step(); gap = tr_b - va_b
        run.log({'train/loss': tr_loss, 'train/bacc': tr_b, 'val/loss': va_loss,
                 'val/bacc': va_b, 'train_val_gap': gap, 'epoch': epoch})
        if va_b > best_val:
            best_val = va_b; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; patience_cnt = 0
        else: patience_cnt += 1
        if patience_cnt >= PATIENCE: log.info(f'  P{subj_id:03d}: val-patience stop @ ep {epoch}'); break
        if gap > GAP_THRESHOLD:
            gap_cnt += 1
            if gap_cnt >= GAP_PATIENCE: log.info(f'  P{subj_id:03d}: gap-stop @ ep {epoch}'); break
        else: gap_cnt = 0
    model.load_state_dict(best_state)
    _, te_b, te_lbl, te_pred = run_epoch(model, te_l)
    torch.save({'state_dict': best_state, 'val_bacc': best_val, 'test_bacc': te_b,
                'labels': te_lbl, 'preds': te_pred}, CKPT_DIR / f'P{subj_id:03d}.pt')
    per_class = recall_score(te_lbl, te_pred, average=None, zero_division=0, labels=list(range(N_CLASSES)))
    run.summary['val_bacc'], run.summary['test_bacc'] = best_val, te_b
    try:
        run.log({'confusion_matrix': wandb.plot.confusion_matrix(
            preds=te_pred.tolist(), y_true=te_lbl.tolist(), class_names=CLASS_NAMES)})
    except Exception: pass
    run.finish()
    return {'val_bacc': best_val, 'test_bacc': te_b, 'labels': te_lbl, 'preds': te_pred}


## §5 — Loop su tutti i soggetti

In [ ]:
SUBJECT_RESULTS = {}
for sid in tqdm(ALL_SUBJ, desc='DHSLP SS 5words'):
    if (CKPT_DIR / f'P{sid:03d}.pt').exists():
        log.info(f'P{sid:03d}: skip (checkpoint presente)'); continue
    loaders = make_loso_loaders(sid)
    if loaders is None:
        log.warning(f'P{sid:03d}: skip (sessioni insufficienti)'); continue
    tr_l, va_l, te_l = loaders
    log.info(f'P{sid:03d}: train={len(tr_l.dataset)} val={len(va_l.dataset)} test={len(te_l.dataset)}')
    try:
        SUBJECT_RESULTS[sid] = train_subject(sid, tr_l, va_l, te_l)
    except Exception as e:
        log.error(f'P{sid:03d}: {e}\n{traceback.format_exc()}')
log.info(f'DONE: {len(SUBJECT_RESULTS)}/{len(ALL_SUBJ)} soggetti')


## §6 — Ranking

In [ ]:
# Ranking dai checkpoint (chance = 20%)
rows = []
for ck in sorted(CKPT_DIR.glob('P*.pt')):
    d = torch.load(ck, weights_only=False)
    rows.append((ck.stem, float(d['test_bacc']), float(d.get('val_bacc', np.nan))))
df = (pd.DataFrame(rows, columns=['Subject', 'Test bAcc', 'Val bAcc'])
        .sort_values('Test bAcc', ascending=False).reset_index(drop=True))
chance = 1 / N_CLASSES
b = df['Test bAcc'].values
print('='*50)
print(f'  V5W_03 — DHSLP Subject-Specific 5words ({len(b)} soggetti)')
print('='*50)
print(f'  mean={b.mean():.4f}  median={np.median(b):.4f}  max={b.max():.4f}  min={b.min():.4f}')
print(f'  chance={chance:.3f}  delta mean-chance={b.mean()-chance:+.4f}')
print(f'  > chance: {(b>chance).sum()}/{len(b)} ({(b>chance).mean()*100:.1f}%)')
print('='*50)
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ca02c' if v > chance else '#d62728' for v in b]
ax.bar(range(len(b)), b, color=colors, alpha=0.85)
ax.axhline(chance, color='k', ls='--', lw=1.5, label=f'Chance ({chance:.0%})')
ax.set_xticks(range(len(b))); ax.set_xticklabels(df['Subject'], rotation=90, fontsize=6)
ax.set_ylabel('Balanced Accuracy'); ax.legend()
ax.set_title('V5W_03 — DHSLP Subject-Specific (5 parole, chance 20%)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'v5w03_dhslp_ss_ranking.png', dpi=150, bbox_inches='tight'); plt.show()
df.to_csv(FIG_DIR / 'v5w03_subject_ranking.csv', index=False)
print('Salvato: v5w03_dhslp_ss_ranking.png + v5w03_subject_ranking.csv')
